# 🛠️ Notebook 1 — LangGraph Tools
### What are Tools, how to define them, and how LLM uses them

**What you'll learn:**
- Why LLMs need Tools
- How to create tools using `@tool` decorator
- How LLM decides to call a tool (`content=''` explained)
- Testing tools directly

---

## 📦 Install Dependencies

In [ ]:
!pip install -qU langgraph langchain langchain-openai requests

In [1]:
from dotenv import load_dotenv
load_dotenv()







True

---
## 🤔 Why do LLMs need Tools?

LLMs are trained on **past data**. They cannot:

| Task | Without Tool | With Tool |
|------|-------------|----------|
| Today's weather | ❌ Guesses | ✅ Calls weather API |
| Precise math | ❌ Might be wrong | ✅ Calls calculator |
| Live stock price | ❌ No idea | ✅ Calls finance API |

**Tools = Python functions that LLM can call when needed.**

```
User: "Weather in Mumbai?"
  → LLM: "I need get_weather tool"
  → Tool runs → returns "32°C, Sunny"
  → LLM: "It is 32°C and sunny in Mumbai!"
```

---
## ✍️ Defining Tools with `@tool`

The `@tool` decorator converts any Python function into a LangChain Tool.

**Rules for good tools:**
1. Always add **type hints** on arguments and return
2. Write a **clear docstring** — LLM reads this to decide when to call the tool
3. Return a **string** — LLM understands text best
4. Handle **exceptions** gracefully

In [2]:
from langchain_core.tools import tool
import requests
import math

# ─────────────────────────────────────────────
# Tool 1: Weather
# ─────────────────────────────────────────────
@tool
def get_weather(location: str) -> str:
    """Get current weather for any city.
    Use this when user asks about weather, temperature, or conditions.
    Args:
        location: City name e.g. 'Mumbai', 'Delhi', 'London'
    """
    url = f"https://wttr.in/{location}?format=j1"
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    data        = response.json()
    current     = data["current_condition"][0]
    temp_c      = current["temp_C"]
    feels_like  = current["FeelsLikeC"]
    humidity    = current["humidity"]
    description = current["weatherDesc"][0]["value"]
    return (
        f"Weather in {location}: {description}, "
        f"Temp: {temp_c}°C, Feels like: {feels_like}°C, "
        f"Humidity: {humidity}%"
    )

# ─────────────────────────────────────────────
# Tool 2: Calculator
# ─────────────────────────────────────────────
@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression safely.
    Use this for any arithmetic, calculations, or math problems.
    Args:
        expression: e.g. '2 + 2', 'sqrt(144)', '15 * 7'
    """
    try:
        allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
        result  = eval(expression, {"__builtins__": {}}, allowed)
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error: {e}"

# ─────────────────────────────────────────────
# Tool 3: Trip Planner
# ─────────────────────────────────────────────
@tool
def get_trip_plan(origin: str, destination: str) -> str:
    """Suggest travel options between two cities.
    Use when user asks how to travel between cities.
    Args:
        origin: Departure city e.g. 'Mumbai'
        destination: Destination city e.g. 'Goa'
    """
    plans = {
        ("mumbai", "goa"): (
            "🚆 Train: Konkan Railway (~8 hrs, scenic)\n"
            "✈️  Flight: ~1 hr (cheapest Tue/Wed)\n"
            "🚗 Road: NH66, ~600 km (~10 hrs)\n"
            "💡 Tip: Book train 60 days in advance!"
        ),
        ("mumbai", "delhi"): (
            "✈️  Flight: ~2 hrs (best option)\n"
            "🚆 Train: Rajdhani Express (~16 hrs overnight)\n"
            "💡 Tip: Rajdhani has great food!"
        ),
        ("delhi", "agra"): (
            "🚄 Gatimaan Express: ~1.5 hrs (fastest)\n"
            "🚗 Yamuna Expressway: ~3 hrs\n"
            "💡 Tip: Visit Taj Mahal at sunrise!"
        ),
    }
    key         = (origin.lower(), destination.lower())
    reverse_key = (destination.lower(), origin.lower())
    plan        = plans.get(key) or plans.get(reverse_key)
    if plan:
        return f"Travel from {origin} to {destination}:\n{plan}"
    return f"Check flights on Skyscanner and trains on IRCTC for {origin} → {destination}."

# ─────────────────────────────────────────────
# Tool 4: Currency Converter
# ─────────────────────────────────────────────
@tool
def currency_converter(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert currency amounts using approximate rates.
    Args:
        amount: Amount to convert e.g. 100
        from_currency: Source currency code e.g. 'USD'
        to_currency: Target currency code e.g. 'INR'
    """
    rates_to_inr = {
        "USD": 83.5, "EUR": 90.2, "GBP": 105.8,
        "AED": 22.7, "SGD": 62.1, "INR": 1.0
    }
    from_c = from_currency.upper()
    to_c   = to_currency.upper()
    if from_c not in rates_to_inr or to_c not in rates_to_inr:
        return f"Unsupported currency. Supported: {list(rates_to_inr.keys())}"
    inr_amount = amount * rates_to_inr[from_c]
    result     = inr_amount / rates_to_inr[to_c]
    return f"{amount} {from_c} = {result:.2f} {to_c} (approximate)"

all_tools = [get_weather, calculate, get_trip_plan, currency_converter]
print("✅ All 4 tools defined!")

✅ All 4 tools defined!


---
## 🧪 Test Tools Directly (without LLM)

In [3]:
# You can call tools directly using .invoke()
print("🌤️  Weather Tool:")
print(get_weather.invoke({"location": "Mumbai"}))

print("\n🧮 Calculator Tool:")
print(calculate.invoke({"expression": "sqrt(144) + 10 * 3"}))

print("\n🗺️  Trip Planner Tool:")
print(get_trip_plan.invoke({"origin": "Mumbai", "destination": "Goa"}))

print("\n💱 Currency Tool:")
print(currency_converter.invoke({"amount": 500, "from_currency": "USD", "to_currency": "INR"}))

🌤️  Weather Tool:
Weather in Mumbai: Haze, Temp: 31°C, Feels like: 35°C, Humidity: 59%

🧮 Calculator Tool:
sqrt(144) + 10 * 3 = 42.0

🗺️  Trip Planner Tool:
Travel from Mumbai to Goa:
🚆 Train: Konkan Railway (~8 hrs, scenic)
✈️  Flight: ~1 hr (cheapest Tue/Wed)
🚗 Road: NH66, ~600 km (~10 hrs)
💡 Tip: Book train 60 days in advance!

💱 Currency Tool:
500.0 USD = 41750.00 INR (approximate)


---
## 🔗 Bind Tools to LLM

`bind_tools()` tells the LLM: *"These tools are available — use them when needed."*

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm            = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
llm_with_tools = llm.bind_tools(all_tools)  # 🔗 register tools

print("✅ Tools bound to LLM!")

✅ Tools bound to LLM!


---
## 💡 Key Insight — `content=''` means Tool Call!

When LLM needs a tool → `content` is **empty**, `tool_calls` has the intent.

This is **normal and expected** — not an error!

In [5]:
print("=" * 55)
print("Case 1: LLM knows the answer directly")
print("=" * 55)
r1 = llm_with_tools.invoke([HumanMessage("What is the capital of France?")])
print(f"content    : {repr(r1.content)}")
print(f"tool_calls : {r1.tool_calls}")

print()
print("=" * 55)
print("Case 2: LLM needs a tool")
print("=" * 55)
r2 = llm_with_tools.invoke([HumanMessage("What is the weather in Mumbai?")])
print(f"content    : {repr(r2.content)}  ← EMPTY = tool call!")
print(f"tool_calls : {r2.tool_calls}")
print(f"\nTool chosen: {r2.tool_calls[0]['name']}")
print(f"Args passed: {r2.tool_calls[0]['args']}")

Case 1: LLM knows the answer directly
content    : 'The capital of France is Paris.'
tool_calls : []

Case 2: LLM needs a tool
content    : ''  ← EMPTY = tool call!
tool_calls : [{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'call_XfjSBaXZKU6t4QdMubKxUlgb', 'type': 'tool_call'}]

Tool chosen: get_weather
Args passed: {'location': 'Mumbai'}


---
## 🏋️ Exercises

1. Add a new tool `get_hotel_price(city: str, nights: int) -> str` that returns a dummy hotel price
2. Call each tool directly with `.invoke()` and print results
3. Bind your new tool to the LLM and check if `tool_calls` is populated when you ask about hotel prices
4. What happens if the tool raises an exception? Add a `try/except` and test it

---
**Next → Notebook 2: Memory** 🧠